In [3]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

print("1. Membaca Data Transaksi dan Produk...")
df_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
df_products = pd.read_csv('../data/raw/olist_products_dataset.csv')

print("2. Menggabungkan Data")
# JOIN berdasarkan product_id untuk mendapatkan nama kategori produknya
df_merge = pd.merge(df_items, df_products, on='product_id')
df_merge = df_merge.dropna(subset=['product_category_name'])

print("3. Membangun Matriks Keranjang Belanja (Basket)")
# Jika dalam 1 order pelanggan beli kategori itu, angkanya 1. Jika tidak, 0.
basket = (df_merge.groupby(['order_id', 'product_category_name'])['order_item_id']
          .count().unstack().fillna(0))
basket_sets = (basket > 0).astype(bool)

print("4. Melatih Algoritma Apriori")
frequent_itemsets = apriori(basket_sets, min_support=0.0001, use_colnames=True)

print("5. Menciptakan Aturan Rekomendasi (Association Rules)")
# Mencari hubungan sebab-akibat antar kategori produk
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

print("\n--- Top Aturan Rekomendasi Cross-Selling ---")
# Menampilkan 5 aturan rekomendasi paling kuat
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

1. Membaca Data Transaksi dan Produk...
2. Menggabungkan Data
3. Membangun Matriks Keranjang Belanja (Basket)
4. Melatih Algoritma Apriori
5. Menciptakan Aturan Rekomendasi (Association Rules)

--- Top Aturan Rekomendasi Cross-Selling ---
                    antecedents                   consequents   support  \
0  frozenset({cama_mesa_banho})    frozenset({casa_conforto})  0.000442   
1    frozenset({casa_conforto})  frozenset({cama_mesa_banho})  0.000442   

   confidence      lift  
0    0.004566  1.118859  
1    0.108312  1.118859  
